In [1]:
!pip install -q sentence-transformers peft datasets faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 81.8 MB/s eta 0:00:00


In [2]:
!pip install -q --upgrade ipython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 624.2/624.2 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.4/85.4 kB 8.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires ipython==7.34.0, but you have ipython 9.11.0 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.2.1 which is incompatible.


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import shutil
from pathlib import Path

try:
  shutil.rmtree(Path("/content/drive/MyDrive/auto-hh/models/BiEncoder"))
except:
  print('BiEncoder ещё нет')

In [5]:
%cd /content/drive/MyDrive/auto-hh/src
%load_ext autoreload
%autoreload 2

/content/drive/MyDrive/auto-hh/src


In [6]:
import torch
from models import BiEncoder, CrossEncoder
from training import Trainer

In [7]:
MODEL_NAME = "BAAI/bge-m3"
USE_LORA = True

model = BiEncoder(
    model_name=MODEL_NAME,
    need_attention=True,
    use_lora=USE_LORA,
)

print(f"Обучаемых параметров: {model.count_trainable_parameters()}")

if model.count_trainable_parameters() > 10000:
    print("LoRA активна")
else:
    print("Параметров слишком мало")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Применена LoRA. Обучаемых параметров: 786432 / 568541184
Обучаемых параметров: 786432
LoRA активна


In [8]:
from lib import create_train_examples

In [9]:
VACANCIES_PATH = "/content/drive/MyDrive/auto-hh/hh_dataset.csv"
RESUMES_PATH = "/content/drive/MyDrive/auto-hh/resumes_generated.csv"

full_dataset = create_train_examples(
    vacancies_path=VACANCIES_PATH,
    resumes_path=RESUMES_PATH,
    max_examples=500
)

Пример данных:
Resume: ВАКАНСИЯ: Frontend Developer (React/TypeScript). ЗАРПЛАТА: 210000 RUB. ОПЫТ: — Frontend Developer в ООО «ТехСервис» (2 г.): Разрабатывал UI‑компоненты на React + TypeScript для CRM‑системы: реестры, формы, модальные окна. Помогал с интеграцией API, писал async thunks, реализовывал optimistic updates. Работал над сложными таблицами с виртуализацией и inline‑редактированием, улучшил производительность рендеринга. Участвовал в code‑review, поддерживал дизайн‑систему MUI, писал тесты в Jest/RTL. — Junior Frontend Engineer в АО «Инновационные Решения» (1 г.): Участвовал в построении дашбордов: графики, виджеты, сохранённые фильтры. Работал над валидацией форм, обрабатывал ошибки загрузки, помогал с настройкой CI/CD веток. Делал небольшие UI‑правки, улучшил пользовательские потоки, иногда писал стили в styled-components.. ОБРАЗОВАНИЕ: Воронежский государственный университет им. М. В. Ломоносова (год выпуска: 2018.0). НАВЫКИ: TypeScript, React, JavaScript, Redux, Git, U

In [10]:
split_dataset = full_dataset.train_test_split(
    test_size=0.2,
    seed=42,
    shuffle=True
)

train_dataset, test_dataset = split_dataset['train'], split_dataset['test']

In [11]:
import torch, gc
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

112

In [12]:
OUTPUT_PATH = "/content/drive/MyDrive/auto-hh/models/BiEncoder"

In [13]:
%%time

trainer = Trainer(
    batch_size=2,
    epochs=2,
    learning_rate=2e-4,
    use_lora=USE_LORA,
)
trainer.train(model, train_dataset, OUTPUT_PATH)


Подготовка данных...
   - Примеров: 400
   - Batch size: 2
   - Epochs: 2
   - LoRA enabled: True

Запуск обучения...


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
10,0.157513
20,0.142784
30,0.102703
40,0.089008
50,0.071515
60,0.028099
70,0.017010
80,0.010123
90,0.009707
100,0.079903


Адаптеры LoRA сохранены в: /content/drive/MyDrive/auto-hh/models/BiEncoder/adapter
CPU times: user 5min 49s, sys: 2min 28s, total: 8min 17s
Wall time: 8min 22s


## 🧪 ПРОВЕРКА ЗАГРУЗКИ И РАБОТЫ БИ ЭНКОДЕРА

In [14]:
!find /content/drive/MyDrive/auto-hh/models/BiEncoder

/content/drive/MyDrive/auto-hh/models/BiEncoder
/content/drive/MyDrive/auto-hh/models/BiEncoder/checkpoint-200
/content/drive/MyDrive/auto-hh/models/BiEncoder/checkpoint-200/config_sentence_transformers.json
/content/drive/MyDrive/auto-hh/models/BiEncoder/checkpoint-200/README.md
/content/drive/MyDrive/auto-hh/models/BiEncoder/checkpoint-200/adapter_model.safetensors
/content/drive/MyDrive/auto-hh/models/BiEncoder/checkpoint-200/adapter_config.json
/content/drive/MyDrive/auto-hh/models/BiEncoder/checkpoint-200/tokenizer_config.json
/content/drive/MyDrive/auto-hh/models/BiEncoder/checkpoint-200/tokenizer.json
/content/drive/MyDrive/auto-hh/models/BiEncoder/checkpoint-200/sentence_bert_config.json
/content/drive/MyDrive/auto-hh/models/BiEncoder/checkpoint-200/1_Pooling
/content/drive/MyDrive/auto-hh/models/BiEncoder/checkpoint-200/1_Pooling/config.json
/content/drive/MyDrive/auto-hh/models/BiEncoder/checkpoint-200/modules.json
/content/drive/MyDrive/auto-hh/models/BiEncoder/checkpoint-20

In [15]:
try:
    loaded_model = BiEncoder.load_trained(
        path=OUTPUT_PATH,
        model_name="BAAI/bge-m3",
        need_attention=True,
        use_lora=True
    )
    loaded_model.eval()

    if torch.cuda.is_available():
        loaded_model = loaded_model.to('cuda')

    print(f"Модель загружена из: {OUTPUT_PATH}")
    print(f"Устройство: {'CUDA' if torch.cuda.is_available() else 'CPU'}")
    print(f"Размерность эмбеддинга: {loaded_model.get_sentence_embedding_dimension()}")
except Exception as e:
    print(f"Ошибка загрузки: {e}")
    raise

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Адаптеры LoRA загружены из: /content/drive/MyDrive/auto-hh/models/BiEncoder/adapter
Модель загружена из: /content/drive/MyDrive/auto-hh/models/BiEncoder
Устройство: CUDA
Размерность эмбеддинга: 1024


In [16]:
import pandas as pd
import torch
from sentence_transformers import util


results = []

with torch.no_grad():
    for item in test_dataset:
        emb_r = loaded_model.encode(item["anchor"], convert_to_tensor=True)
        emb_v = loaded_model.encode(item["positive"], convert_to_tensor=True)

        score = util.cos_sim(emb_r, emb_v).item()

        results.append({
            "anchor": item["anchor"],
            "positive": item["positive"],
            "score": score
        })

results_df = pd.DataFrame(results)
results_df.to_csv("test_results.csv", index=False)

print(f"Сохранено {len(results)} результатов в test_results.csv")
print(f"\nСтатистика:")
print(f"Средний скор: {results_df['score'].mean():.4f}")
print(f"Мин/Макс: {results_df['score'].min():.4f} / {results_df['score'].max():.4f}")
print(f"Медиана: {results_df['score'].median():.4f}")

print(f"\nПервые 10 результатов:")
print(results_df.head(10))

Сохранено 100 результатов в test_results.csv

Статистика:
Средний скор: 0.8325
Мин/Макс: 0.6234 / 0.9446
Медиана: 0.8451

Первые 10 результатов:
                                              anchor  \
0  ВАКАНСИЯ: Senior DevOps Engineer. ЗАРПЛАТА: 46...   
1  ВАКАНСИЯ: Бариста. ЗАРПЛАТА: 80000 RUB. ОПЫТ: ...   
2  ВАКАНСИЯ: Официант. ЗАРПЛАТА: 80000 RUB. ОПЫТ:...   
3  ВАКАНСИЯ: Product Manager. ЗАРПЛАТА: 210000 RU...   
4  ВАКАНСИЯ: Java-разработчик. ЗАРПЛАТА: 520000 R...   
5  ВАКАНСИЯ: Senior Data Scientist / ML Engineer....   
6  ВАКАНСИЯ: Системный аналитик 1С. ЗАРПЛАТА: 220...   
7  ВАКАНСИЯ: C/C++ Developer. ЗАРПЛАТА: 260000 RU...   
8  ВАКАНСИЯ: Бариста-кассир. ЗАРПЛАТА: 70 RUB. ОП...   
9  ВАКАНСИЯ: Junior Golang разработчик. ЗАРПЛАТА:...   

                                            positive     score  
0  ВАКАНСИЯ: Senior DevOps Engineer. ЗАРПЛАТА: Не...  0.815718  
1  ВАКАНСИЯ: Бариста. ЗАРПЛАТА: Не указано. ОПЫТ:...  0.779733  
2  ВАКАНСИЯ: Официант. ЗАРПЛАТА: Не указано

In [17]:
from sentence_transformers import SentenceTransformer, util
import torch

test_resume = """
Python разработчик с опытом 5 лет.
Стек: Django, FastAPI, PostgreSQL, Docker, Kubernetes.
Опыт работы с микросервисами и Kafka.
"""

test_vacancy_match = """
Требуется Senior Python разработчик.
Стек: Django, FastAPI, PostgreSQL.
Опыт с микросервисами и очередями сообщений.
"""

test_vacancy_mismatch = """
Требуется Java разработчик.
Стек: Spring, Hibernate, MySQL.
Опыт работы с корпоративными системами.
"""

emb_resume = loaded_model.encode(test_resume, convert_to_tensor=True)
emb_match = loaded_model.encode(test_vacancy_match, convert_to_tensor=True)
emb_mismatch = loaded_model.encode(test_vacancy_mismatch, convert_to_tensor=True)

sim_match = util.cos_sim(emb_resume, emb_match).item()
sim_mismatch = util.cos_sim(emb_resume, emb_mismatch).item()

print(f"Релевантная пара: {sim_match:.4f}")
print(f"Нерелевантная пара: {sim_mismatch:.4f}")
print(f"Разница: {sim_match - sim_mismatch:.4f}")

if sim_match > sim_mismatch:
    print("Модель работает корректно!")
else:
    print("Модель требует дообучения")
    print("Нерелевантная вакансия имеет больший скор")

if sim_match > 0.5:
    print("Хорошее сходство для релевантной пары")
else:
    print("Низкое сходство (возможно, мало данных для обучения)")

Релевантная пара: 0.8496
Нерелевантная пара: 0.4966
Разница: 0.3530
Модель работает корректно!
Хорошее сходство для релевантной пары


In [18]:
print("\n⚡ ТЕСТ СКОРОСТИ:")

import time

n_requests = 10
start = time.time()
for _ in range(n_requests):
    loaded_model.encode([test_resume], convert_to_numpy=True)
end = time.time()

avg_time = (end - start) / n_requests * 1000  # мс
print(f"   Среднее время на запрос: {avg_time:.2f} мс")
print(f"   Запросов в секунду: {1000/avg_time:.1f}")

# ────────────────────────────────────────
print("\n" + "=" * 60)
print("🎉 ТЕСТИРОВАНИЕ ЗАВЕРШЕНО")
print("=" * 60)


⚡ ТЕСТ СКОРОСТИ:
   Среднее время на запрос: 33.00 мс
   Запросов в секунду: 30.3

🎉 ТЕСТИРОВАНИЕ ЗАВЕРШЕНО


In [19]:
print("\nТест извлечения весов внимания")
weights = loaded_model.get_attention_weights(test_resume)

top_tokens = sorted(weights.items(), key=lambda x: x[1], reverse=True)[:10]
print("Топ токенов по важности:")
for token, weight in top_tokens:
    print(f"  {token}: {weight:.4f}")


Тест извлечения весов внимания


`sdpa` attention does not support `output_attentions=True`. Please set your attention to `eager` if you want any of these features.


ValueError: Attention weights = None. Проверь output_attentions=True

## ТЕСТ КРОСС ЭНКОДЕРА

In [ ]:
# ─── Загрузка ──────────────────────────
print("📥 Загрузка CrossEncoder...")
ce = CrossEncoder("BAAI/bge-reranker-v2-m3")

In [ ]:
query = "Python разработчик Django"
candidates = [
    "Требуется Python разработчик с опытом Django и FastAPI",
    "Вакансия: Java разработчик Spring Boot",
    "Ищем DevOps инженера Kubernetes Docker",
]

scores = ce.rerank(query, candidates)
print(*scores, sep='\n')

# ─── Проверка ──────────────────────────
assert len(scores) == 3, "❌ Количество scores не совпадает!"
assert 0 <= scores[0]["score"] <= 1, "❌ Score вне диапазона!"
print("\n✅ CrossEncoder работает! 🎉")